# Ejemplos con Hive

## Descripción de las variables

El dataset, obtenido de <a target = "_blank" href="https://www.transtats.bts.gov/Fields.asp?Table_ID=236">este link</a> está compuesto por las siguientes variables referidas siempre al año 2018:

1. **Month** 1-4
2. **DayofMonth** 1-31
3. **DayOfWeek** 1 (Monday) - 7 (Sunday)
4. **FlightDate** fecha del vuelo
5. **Origin** código IATA del aeropuerto de origen
6. **OriginCity** ciudad donde está el aeropuerto de origen
7. **Dest** código IATA del aeropuerto de destino
8. **DestCity** ciudad donde está el aeropuerto de destino  
9. **DepTime** hora real de salida (local, hhmm)
10. **DepDelay** retraso a la salida, en minutos
11. **ArrTime** hora real de llegada (local, hhmm)
12. **ArrDelay** retraso a la llegada, en minutos: se considera que un vuelo ha llegado "on time" si aterrizó menos de 15 minutos más tarde de la hora prevista en el Computerized Reservations Systems (CRS).
13. **Cancelled** si el vuelo fue cancelado (1 = sí, 0 = no)
14. **CancellationCode** razón de cancelación (A = aparato, B = tiempo atmosférico, C = NAS, D = seguridad)
15. **Diverted** si el vuelo ha sido desviado (1 = sí, 0 = no)
16. **ActualElapsedTime** tiempo real invertido en el vuelo
17. **AirTime** en minutos
18. **Distance** en millas
19. **CarrierDelay** en minutos: El retraso del transportista está bajo el control del transportista aéreo. Ejemplos de sucesos que pueden determinar el retraso del transportista son: limpieza de la aeronave, daño de la aeronave, espera de la llegada de los pasajeros o la tripulación de conexión, equipaje, impacto de un pájaro, carga de equipaje, servicio de comidas, computadora, equipo del transportista, problemas legales de la tripulación (descanso del piloto o acompañante) , daños por mercancías peligrosas, inspección de ingeniería, abastecimiento de combustible, pasajeros discapacitados, tripulación retrasada, servicio de inodoros, mantenimiento, ventas excesivas, servicio de agua potable, denegación de viaje a pasajeros en mal estado, proceso de embarque muy lento, equipaje de mano no válido, retrasos de peso y equilibrio.
20. **WeatherDelay** en minutos: causado por condiciones atmosféricas extremas o peligrosas, previstas o que se han manifestado antes del despegue, durante el viaje, o a la llegada.
21. **NASDelay** en minutos: retraso causado por el National Airspace System (NAS) por motivos como condiciones meteorológicas (perjudiciales pero no extremas), operaciones del aeropuerto, mucho tráfico aéreo, problemas con los controladores aéreos, etc.
22. **SecurityDelay** en minutos: causado por la evacuación de una terminal, re-embarque de un avión debido a brechas en la seguridad, fallos en dispositivos del control de seguridad, colas demasiado largas en el control de seguridad, etc.
23. **LateAircraftDelay** en minutos: debido al propio retraso del avión al llegar, problemas para conseguir aterrizar en un aeropuerto a una hora más tardía de la que estaba prevista.

Leemos el fichero CSV utilizando el delimitador por defecto de Spark (","). La primera línea contiene encabezados (nombres de columnas) por lo que no es parte de los datos y debemos indicarlo con la opción header.

## 1. Leemos los datos del fichero CSV que hay en Azure Data Lake Storage

In [38]:
import os

from IPython.core.displaypub import DisplayPublisher
from databricks.connect import DatabricksSession
from dotenv import load_dotenv

# Initialize Spark session and environment variables
spark = DatabricksSession.builder.getOrCreate()
load_dotenv()
STORAGE_ACCOUNT_NAME = os.getenv("STORAGE_ACCOUNT_NAME")
AZURE_ACCOUNT_KEY = os.getenv("AZURE_ACCOUNT_KEY")

In [39]:
# Esto no hace nada: la lectura es lazy así que no se lee en realidad hasta que ejecutemos una acción sobre flightsDF
# Solamente se comprueba que exista el fichero en esa ruta, y se leen los nombres de columnas
flightsDF = (spark.read
                 .option(f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", AZURE_ACCOUNT_KEY)\
                 .option("header", "true")\
                 .csv(f"abfss://datos@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/flights-jan-apr-2018.csv"))

## 2. Vamos a mostrar el contenido del metastore de Hive, que ahora mismo no tiene nada

In [3]:
# Por usar databric-connect free edition
spark.sql("USE CATALOG workspace")

DataFrame[]

In [4]:
spark.sql("show tables").show()

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
+--------+---------+-----------+



## 3. Creamos una vista temporal de un DF que tiene solo dos columnas

Esto solamente añade metadatos al metastore de Hive, que además se borrarán cuando cerremos el notebook. No guarda datos físicos de la tabla en ningún lado, puesto que el DF está en memoria. O mejor dicho, el DF del que proviene esta vista temporal "no está en ningún lado", porque no hemos cacheado weatherDistanceDF así que cualquier consulta SQL que hagamos sobre la tabla `weatherDistanceTable` provoca que se tenga que re-calcular el DF `weatherDistanceDF` sobre el cual está creada dicha tabla temporal.

In [5]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

weatherDistanceDF = (flightsDF
                     .select("WeatherDelay", "Distance")
                     .withColumn("WeatherDelay", col("WeatherDelay").cast(DoubleType()))
                     .withColumn("Distance", col("Distance").cast(DoubleType()))
                     )
weatherDistanceDF.createOrReplaceTempView("weatherDistanceTable")

Ahora vemos que la tabla se ha creado como vista temporal. Registrar un DF como vista temporal no requiere en absoluto que el DF esté materializado.

In [6]:
spark.sql("show tables").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|        |weatherdistancetable|       true|
+--------+--------------------+-----------+



Podemos hacer consultas sobre ella

In [7]:
flightsLejosDF = spark.sql("select * from weatherDistanceTable where Distance > 200 and WeatherDelay is not null limit 5")
flightsLejosDF.show()

+------------+--------+
|WeatherDelay|Distance|
+------------+--------+
|         8.0|   395.0|
|        14.0|   395.0|
|         0.0|   395.0|
|        16.0|   395.0|
|        20.0|   395.0|
+------------+--------+



Cacheamos la tabla con una operación que **sí la materializa inmediatamente**. El comando CACHE TABLE es similar a cache() pero sí actúa de manera inmediata.

In [8]:
# No supported with serverless
# spark.sql("CACHE TABLE weatherDistanceTable")

## 4. Creamos una tabla persistente manejada, guardando como tabla el contenido de un DF

Primero vamos a crear una database específica que no sea la default, que es la única database pre-existente. Para no tener problemas con la escritura de datos en ADLS asociados a tablas externas (como haremos posteriormente), vamos a utilizar siempre el `hive_metastore`, en lugar del catálogo de Unity que viene ya creado por defecto y que se llama igual que nuestra plataforma de Databricks. La escritura en ADLS asociando los ficheros a tablas externas requiere crear una `EXTERNAL LOCATION` apuntando a una ruta de ADLS, que es un proceso un poco más laborioso.

In [41]:
spark.sql("USE CATALOG workspace")

DataFrame[]

In [42]:
spark.sql("create database if not exists vuelos")

DataFrame[]

In [43]:
spark.sql("show tables").show()

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
+--------+---------+-----------+



In [48]:
flightsJFK = flightsDF.where("Origin = 'JFK'")#.cache()
# flightsJFK.write.saveAsTable("vuelos.flightsjfk") # es una acción: se guardan físicamente los datos en una ubicación de DBFS configurada para las tablas manejadas

Si volvemos a mostrar las tablas que existen, veremos la nueva. Vemos que **no** es temporal, pero no sabemos si es manejada o es externa.

In [44]:
spark.sql("show tables").show()  # esto muestra las tablas de la database activa, que actualmente es default

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
+--------+---------+-----------+



In [45]:
spark.sql("show tables in vuelos").show()  # muestra las tablas de la database vuelos. Podemos omitir esto si fijamos la database vuelos como la activa

+--------+----------+-----------+
|database| tableName|isTemporary|
+--------+----------+-----------+
|  vuelos|flightsjfk|      false|
+--------+----------+-----------+



In [46]:
spark.sql("use database vuelos")
(spark.sql("show tables").show())   # de aquí en adelante no será necesario indicar la database

+--------+----------+-----------+
|database| tableName|isTemporary|
+--------+----------+-----------+
|  vuelos|flightsjfk|      false|
+--------+----------+-----------+



In [16]:
spark.sql("describe extended vuelos.flightsjfk").show()

+-----------------+---------+-------+
|         col_name|data_type|comment|
+-----------------+---------+-------+
|            Month|   string|   NULL|
|       DayofMonth|   string|   NULL|
|        DayOfWeek|   string|   NULL|
|       FlightDate|   string|   NULL|
|           Origin|   string|   NULL|
|       OriginCity|   string|   NULL|
|             Dest|   string|   NULL|
|         DestCity|   string|   NULL|
|          DepTime|   string|   NULL|
|         DepDelay|   string|   NULL|
|          ArrTime|   string|   NULL|
|         ArrDelay|   string|   NULL|
|        Cancelled|   string|   NULL|
| CancellationCode|   string|   NULL|
|         Diverted|   string|   NULL|
|ActualElapsedTime|   string|   NULL|
|          AirTime|   string|   NULL|
|         Distance|   string|   NULL|
|     CarrierDelay|   string|   NULL|
|     WeatherDelay|   string|   NULL|
+-----------------+---------+-------+
only showing top 20 rows


* Como vemos, el campo Location indica la carpeta donde se están guardando las tablas *manejadas*, que es la carpeta /user/hive/warehouse/<nombretabla> de DBFS. 
* Por defecto el formato de fichero en Databricks es Delta

In [17]:
!ls /dbfs/user/hive/warehouse/vuelos.db/flightsjfk -l

'ls' is not recognized as an internal or external command,
operable program or batch file.


## 5. Guardamos flightsDF como fichero Delta en ADLS, sin crear ninguna tabla en el metastore

Lo vamos a guardar particionado por la columna Origin. Como este DF sólo tiene dos aeropuertos distintos porque hemos retenido solamente los vuelos que salen de SFO o de LAX, Spark creará dos subcarpetas. Dentro de cada subcarpeta habrá tantos ficheros como particiones tiene el DF, que actualmente son 3. Se puede comprobar con `flightsSFO.rdd.getNumPartitions()`

In [49]:
flightsSFO = flightsDF.where("Origin = 'SFO' or Origin = 'LAX'")\
                      .select("FlightDate", "Origin", "Dest", "Distance")

(flightsSFO.write.option(f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", AZURE_ACCOUNT_KEY)
                .mode("overwrite")\
                .partitionBy("Origin")\
                .format("delta")\
                .save(f"abfss://datos@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/flightsSFO"))

In [50]:
from databricks.sdk.runtime.dbutils_stub import dbutils

(dbutils.fs.ls(f"abfss://datos@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/flightsSFO"))

In [25]:
flightsSFO.rdd.getNumPartitions()

PySparkNotImplementedError: [NOT_IMPLEMENTED] rdd is not implemented.

In [28]:
read_sfo_df = (spark.read
               .option("format", "delta")
               .option(f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", AZURE_ACCOUNT_KEY)
               .load(f"abfss://datos@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/flightsSFO" )
               )
read_sfo_df.limit(10).show()

+----------+------+----+--------+
|FlightDate|Origin|Dest|Distance|
+----------+------+----+--------+
|2018-01-01|   SFO| JFK| 2586.00|
|2018-01-01|   SFO| FLL| 2584.00|
|2018-01-01|   SFO| BOS| 2704.00|
|2018-01-01|   SFO| JFK| 2586.00|
|2018-01-01|   SFO| BOS| 2704.00|
|2018-01-01|   SFO| FLL| 2584.00|
|2018-01-01|   SFO| LGB|  354.00|
|2018-01-01|   SFO| BOS| 2704.00|
|2018-01-01|   SFO| JFK| 2586.00|
|2018-01-01|   SFO| LGB|  354.00|
+----------+------+----+--------+



## 6. Ahora vamos a crear una tabla EXTERNA a partir del fichero existente /flightsSFO (que en realidad es una carpeta)

### Hay varias maneras de crear una tabla externa. Aquí vamos a crear una tabla externa a partir de un fichero (carpeta) que ya existe. Más adelante veremos otra manera.

### 6.1. Primera manera (RECOMENDADA): no indicar el esquema pero indicar `using delta location <ruta>` y la tabla automáticamente se creará con el esquema de ese fichero Delta

In [ ]:
https://masterjcbt001sta.blob.core.windows.net/datos/flightsSFO/

In [30]:
LOCATION = f"abfss://datos@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/flightsSFO"
spark.sql(f"create table vuelos.flightssfo using delta location '{LOCATION}'")

AnalysisException: [NO_PARENT_EXTERNAL_LOCATION_FOR_PATH] No parent external location was found for path 'abfss://datos@masterjcbt001sta.blob.core.windows.net/datos/flightsSFO'. Please create an external location on one of the parent paths and then retry the query or command again. SQLSTATE: 22KD1

JVM stacktrace:
org.apache.spark.sql.catalyst.analysis.NoParentExternalLocationForPathException
	at com.databricks.sql.managedcatalog.client.ManagedCatalogClientImpl.$anonfun$generateTemporaryPathCredentials$1(ManagedCatalogClientImpl.scala:7065)
	at com.databricks.sql.managedcatalog.client.ManagedCatalogClientImpl.$anonfun$recordAndWrapExceptionBase$2(ManagedCatalogClientImpl.scala:8287)
	at com.databricks.spark.util.FrameProfiler$.$anonfun$record$1(FrameProfiler.scala:114)
	at com.databricks.spark.util.FrameProfilerExporter$.maybeExportFrameProfiler(FrameProfilerExporter.scala:203)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:105)
	at com.databricks.sql.managedca

### Comprobamos que ahora tenemos una tabla más, persistente y con idéntico esquema, llamada flightsSFO

In [0]:
spark.sql("show tables").show(truncate = False)

+--------+--------------------+-----------+
|database|tableName           |isTemporary|
+--------+--------------------+-----------+
|vuelos  |flightsjfk          |false      |
|vuelos  |flightssfo          |false      |
|        |weatherdistancetable|true       |
+--------+--------------------+-----------+



### ¿La tabla flightsSFO es externa, o es manejada?

In [0]:
spark.sql("describe extended vuelos.flightssfo").display()

col_name,data_type,comment
FlightDate,string,null
Origin,string,null
Dest,string,null
Distance,string,null
# Partition Information,,
# col_name,data_type,comment
Origin,string,null
,,
# Delta Statistics Columns,,
Column Names,"FlightDate, Dest, Distance",


### 6.2 Segunda manera de crear una tabla externa: en una misma operación guardamos nuevo el DF en otra ubicación y creamos al mismo tiempo una tabla externa sobre dichos datos


* El detalle para que la tabla sea creada como EXTERNA es indicar `.option("path", "...")` antes de `saveAsTable`. 
* Vamos a guardar el DF en la carpeta flights de ADLS

In [ ]:
(flightsSFO.write.option(f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", AZURE_ACCOUNT_KEY)
                .mode("overwrite")\
                .partitionBy("Origin")\
                .format("delta")\
                .save(f"abfss://datos@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/flightsSFO"))

In [56]:
flightsJFK.select("FlightDate", "Origin", "Dest", "Distance").show()

+----------+------+----+--------+
|FlightDate|Origin|Dest|Distance|
+----------+------+----+--------+
|2018-01-01|   JFK| FLL| 1069.00|
|2018-01-01|   JFK| SJU| 1598.00|
|2018-01-01|   JFK| BOS|  187.00|
|2018-01-01|   JFK| BTV|  266.00|
|2018-01-01|   JFK| ABQ| 1826.00|
|2018-01-01|   JFK| SLC| 1990.00|
|2018-01-01|   JFK| RNO| 2411.00|
|2018-01-01|   JFK| SAN| 2446.00|
|2018-01-01|   JFK| DEN| 1626.00|
|2018-01-01|   JFK| BUF|  301.00|
|2018-01-01|   JFK| ORD|  740.00|
|2018-01-01|   JFK| PWM|  273.00|
|2018-01-01|   JFK| SYR|  209.00|
|2018-01-01|   JFK| BOS|  187.00|
|2018-01-01|   JFK| LAX| 2475.00|
|2018-01-01|   JFK| PHX| 2153.00|
|2018-01-01|   JFK| PSP| 2378.00|
|2018-01-01|   JFK| PBI| 1028.00|
|2018-01-01|   JFK| SMF| 2521.00|
|2018-01-01|   JFK| SRQ| 1041.00|
+----------+------+----+--------+
only showing top 20 rows


In [55]:
(flightsJFK.select("FlightDate", "Origin", "Dest", "Distance")
          .write.option(f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", AZURE_ACCOUNT_KEY)
          .option("path", f"abfss://datos@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/flights")
          .saveAsTable("vuelos.flightsjfk_externa"))

AnalysisException: [NO_PARENT_EXTERNAL_LOCATION_FOR_PATH] No parent external location was found for path 'abfss://datos@masterjcbt001sta.dfs.core.windows.net/flights'. Please create an external location on one of the parent paths and then retry the query or command again. SQLSTATE: 22KD1

JVM stacktrace:
org.apache.spark.sql.catalyst.analysis.NoParentExternalLocationForPathException
	at com.databricks.sql.managedcatalog.client.ManagedCatalogClientImpl.$anonfun$generateTemporaryPathCredentials$1(ManagedCatalogClientImpl.scala:7065)
	at com.databricks.sql.managedcatalog.client.ManagedCatalogClientImpl.$anonfun$recordAndWrapExceptionBase$2(ManagedCatalogClientImpl.scala:8287)
	at com.databricks.spark.util.FrameProfiler$.$anonfun$record$1(FrameProfiler.scala:114)
	at com.databricks.spark.util.FrameProfilerExporter$.maybeExportFrameProfiler(FrameProfilerExporter.scala:203)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:105)
	at com.databricks.sql.managedcatalog.clie

In [0]:
spark.sql("show tables").show(truncate=False)

+--------+--------------------+-----------+
|database|tableName           |isTemporary|
+--------+--------------------+-----------+
|vuelos  |flightsjfk          |false      |
|vuelos  |flightsjfk_externa  |false      |
|vuelos  |flightssfo          |false      |
|        |weatherdistancetable|true       |
+--------+--------------------+-----------+



Comprobamos que efectivamente se ha creado como tabla externa:

In [0]:
spark.sql("describe extended vuelos.flightsjfk_externa").display()

col_name,data_type,comment
FlightDate,string,null
Origin,string,null
Dest,string,null
Distance,string,null
,,
# Delta Statistics Columns,,
Column Names,"FlightDate, Origin, Dest, Distance",
Column Selection Method,first-32,
,,
# Detailed Table Information,,


## Vamos a comprobar el efecto de guardar un DF en una carpeta ya existente, sobreescribiendo solamente las particiones de los valores que estén presentes en los datos que ahora estamos guardando

Como flightsJFK sólo tiene Origin el aeropuerto JFK, podríamos guardarlo en la misma ubicación que flightsSFO y pedirle `.mode("overwrite").partitionBy("Origin")` y esa operación NO borrará lo que ya había sino que solamente reemplazará las particiones que ya existieran. Como la única partición que ahora se va a crear es la de JFK y esa no existía, el resultado es que simplemente se añade una partición más (es decir una subcarpeta Origin=JFK) a la carpeta.

**IMPORTANTE**: el DataFrame que vamos a guardar particionado para sobreescribir ciertas particiones debe tener exactamente EL MISMO ESQUEMA que los datos de las particiones ya existentes.

**IMPORTANTE**: esta operación funciona cuando el parámetro de Spark llamado `"spark.sql.sources.partitionOverwriteMethod"` está fijado al valor `"dynamic"`, o bien cuando indicamos en la propia operación de escritura dicho parámetro con la opción `option("partitionOverwriteMode", "dynamic")`.

El método habitual para dar valor a los parámetros es llamar a `spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")`. Podemos consultar qué valor tiene actualmente con
`spark.conf.get("spark.sql.sources.partitionOverwriteMode")` y veremos que está configurado como `STATIC`.

Aunque en este ejemplo en realidad hemos "añadido" una nueva partición porque los datos que vamos a guardar la segunda vez no contienen ningún origen ya existente (solamente un Origin *nuevo* que es JFK), ocurriría exactamente lo mismo si los datos sí tuviesen algún aeropuerto de particiones ya existentes: se reemplazaría completa la partición de ese aeropuerto ya existente. Si además hubiese aeropuertos que todavía no existían y no tenían subcarpeta, se crearían nuevas particiones (nuevas subcarpetas) para dichos aeropuertos, sin tocar las subcarpetas existentes.

In [57]:
spark.conf.get("spark.sql.sources.partitionOverwriteMode")

AnalysisException: [CONFIG_NOT_AVAILABLE] Configuration spark.sql.sources.partitionOverwriteMode is not available. SQLSTATE: 42K0I

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at com.databricks.sql.connect.SparkConnectConfig$.assertConfigAllowedForRead(SparkConnectConfig.scala:315)
	at org.apache.spark.sql.connect.service.SparkConnectConfigHandler$RuntimeConfigWrapper.get(SparkConnectConfigHandler.scala:113)
	at org.apache.spark.sql.connect.service.SparkConnectConfigHandler.transform(SparkConnectConfigHandler.scala:285)
	at org.apache.spark.sql.connect.service.SparkConnectConfigHandler.$anonfun$handleGet$1(SparkConnectConfigHandler.scala:322)
	at org.apache.spark.sql.connect.service.SparkConnectConfigHandler.$anonfun$handleGet$1$adapted(SparkConnectConfigHandler.scala:321)
	at scala.collection.IterableOnceOps.foreach(IterableOnce.scala:619)
	at scala.collection.IterableOnceOps.foreach$(IterableOnce.scala:617)
	at scala.collection.AbstractIterator.foreach(Iterator.scala:1306

In [0]:
# IMPRESCINDIBLE para que sólo se reemplace en la carpeta las particiones que estemos escribiendo (recordar poner partitionBy porque en caso contrario se reemplaza la carpeta completa!)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

# IMPRESCINDIBLE para que el DataFrame que vamos a guardar tenga el mismo esquema que los datos ya existentes
flightsJFK.select("FlightDate", "Origin", "Dest", "Distance")\
          .write\
          .mode("overwrite")\
          .partitionBy("Origin")\
          .format("delta")\
          .save("abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO")

### Comprobamos que ha añadido una subcarpeta `Origin=JFK` y no ha borrado nada de lo que había en esa carpeta

In [0]:
display(dbutils.fs.ls("abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO"))

path,name,size,modificationTime
abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO/Origin=JFK/,Origin=JFK/,0,1707125477000
abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO/Origin=LAX/,Origin=LAX/,0,1707120148000
abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO/Origin=SFO/,Origin=SFO/,0,1707120149000
abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO/_delta_log/,_delta_log/,0,1707120145000


### Probamos la opción `replaceWhere` específico del formato Delta

Permite reemplazar filas de acuerdo a cierta condición, tanto si la condición se refiere a columnas por las cuales está particionada la carpeta como si hace referencia a otras columnas.

In [0]:
nuevo_df = spark.createDataFrame(
    [("2024-02-01", "LAX", "GRX", "6000"),
     ("2024-02-01", "SFO", "GRX", "5500"),
     ("2024-02-01", "LAS", "GRX", "6200")], 
    ["FlightDate", "Origin", "Dest", "Distance"])

In [0]:
nuevo_df.display()

FlightDate,Origin,Dest,Distance
2024-02-01,LAX,GRX,6000
2024-02-01,SFO,GRX,5500
2024-02-01,LAS,GRX,6200


**La siguiente escritura fallará:** Para poder escribir un DF que contiene datos con más valores que los indicados en replaceWhere, hay que indicar que no valide la constraint que por defecto comprueba, que consiste en que el DF debe contener exclusivamente filas cuyo valor esté contemplado en la condición de replaceWhere

In [0]:
# No necesitamos indicarle cómo está particionada la carpeta donde estamos escribiendo

nuevo_df.write\
        .mode("overwrite")\
        .option("replaceWhere", "Origin = 'SFO'")\
        .format("delta")\
        .save("abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-2247484064188302>, line 7
      1 # No necesitamos indicarle cómo está particionada la carpeta donde estamos escribiendo
      3 nuevo_df.write\
      4         .mode("overwrite")\
      5         .option("replaceWhere", "Origin = 'SFO'")\
      6         .format("delta")\
----> 7         .save("abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO")

File /databricks/spark/python/pyspark/instrumentation_utils.py:47, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     45 start = time.perf_counter()
     46 try:
---> 47     res = func(*args, **kwargs)
     48     logger.log_success(
     49         module_name, class_name, function_name, time.perf_counter() - start, signature
     50     )
     51     return res

File /databricks/spark/python/pyspark/sql/readwriter.py:1681, in DataFrameWriter.save(self, path, for

**Desactivamos la constraint y probamos de nuevo la escritura**

* **IMPORTANTE**: no podemos especificar la opción ad-hoc `option("partitionOverwriteMode", "dynamic")` y la opción `option("replaceWhere", "...")` en una misma operación de escritura. 
* Sí está permitido utilizar replaceWhere cuando el particionamiento dinámico está configurado como parámetro de Spark, puesto que dicho parámetro será ignorado, y el comportamiento lo marcará replaceWhere

In [0]:
spark.conf.set("spark.databricks.delta.replaceWhere.constraintCheck.enabled", False)  # desactivar la restricción anterior

nuevo_df.write\
        .mode("overwrite")\
        .option("replaceWhere", "Origin = 'SFO'")\
        .format("delta")\
        .save("abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO")

### Comprobamos qué ha provocado el guardado anterior

El reemplazamiento estaba indicado solamente para las filas de SFO. Debería haber ocurrido lo siguiente:
* Debe haberse añadido (a pesar de que no fue necesario especificar particionamiento) una nueva subcarpeta (una nueva partición) para el nuevo origen LAS que no existía.
* Debería haberse añadido la fila de la partición existente LAX, puesto que dicho valor no se incluyó en el replaceWhere pero como hay un dato, debe añadirse.
* Debería haberse reemplazado el contenido de toda la subcarpeta de SFO por una única fila, el vuelo con Origin=LAX y Dest=GRX, y por tanto habrá desaparecido todo lo que había en esa partición y ahora habrá una sola fila.

Consultamos tanto la estructura de carpetas como el contenido del DF que podemos leer.
#### Recordemos que sobre la carpeta flightsSFO, una vez escrita, habíamos definido la tabla externa `vuelos.flightssfo` a posteriori

In [0]:
flights_read = spark.read.table("vuelos.flightssfo")
# también podríamos: flights_read = spark.read.format("delta").load("abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO")
flights_read.where("FlightDate = '2024-02-01'").display()

FlightDate,Origin,Dest,Distance
2024-02-01,LAX,GRX,6000
2024-02-01,SFO,GRX,5500
2024-02-01,LAS,GRX,6200


In [0]:
flights_read.where("Origin = 'LAX'").sort("FlightDate", ascending=False).limit(5).display()
flights_read.where("Origin = 'SFO'").sort("FlightDate", ascending=False).limit(5).display()

FlightDate,Origin,Dest,Distance
2024-02-01,LAX,GRX,6000
2018-04-30,LAX,JFK,2475.00
2018-04-30,LAX,KOA,2504.00
2018-04-30,LAX,JFK,2475.00
2018-04-30,LAX,JFK,2475.00


FlightDate,Origin,Dest,Distance
2024-02-01,SFO,GRX,5500


## 7. Borramos tabla externa y comprobamos que la carpeta sigue ahí

In [0]:
spark.sql("drop table vuelos.flightssfo")
spark.sql("show tables").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|  vuelos|          flightsjfk|      false|
|  vuelos|  flightsjfk_externa|      false|
|        |weatherdistancetable|       true|
+--------+--------------------+-----------+



### Comprobamos que sigue existiendo la carpeta /flightsSFO

In [0]:
display(dbutils.fs.ls("abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO"))

path,name,size,modificationTime
abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO/Origin=JFK/,Origin=JFK/,0,1707125477000
abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO/Origin=LAS/,Origin=LAS/,0,1707125760000
abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO/Origin=LAX/,Origin=LAX/,0,1707120148000
abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO/Origin=SFO/,Origin=SFO/,0,1707120149000
abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO/_delta_log/,_delta_log/,0,1707120145000


## 8. Comprobamos dónde están los datos de la tabla persistente manejada que habíamos guardado con saveAsTable

In [0]:
!ls -l /dbfs/user/hive/warehouse/

total 4
drwxrwxrwx 2 nobody nogroup 4096 Feb  5 09:03 vuelos.db


In [0]:
!ls -l /dbfs/user/hive/warehouse/vuelos.db/flightsjfk

total 519
drwxrwxrwx 2 nobody nogroup   4096 Feb  5 09:03 _delta_log
-rwxrwxrwx 1 nobody nogroup 136349 Feb  5 07:55 part-00000-143b3899-1d97-4a87-9f59-34f5e4cfeac6-c000.snappy.parquet
-rwxrwxrwx 1 nobody nogroup 120778 Feb  5 07:55 part-00001-a2b370e9-a068-46e7-90b0-d7b862bd2483-c000.snappy.parquet
-rwxrwxrwx 1 nobody nogroup 153916 Feb  5 07:55 part-00002-9096108e-03c7-4b82-9b7f-3a2475a4aaa7-c000.snappy.parquet
-rwxrwxrwx 1 nobody nogroup 114997 Feb  5 07:55 part-00003-190c3d46-e662-43d2-8345-4d809bf9e3d8-c000.snappy.parquet


## 9. Borramos la tabla manejada y comprobamos que, al ser manejada, Spark ha borrado físicamente esos datos al borrar la tabla

In [0]:
spark.sql("drop table vuelos.flightsjfk")
spark.sql("show tables").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|  vuelos|  flightsjfk_externa|      false|
|        |weatherdistancetable|       true|
+--------+--------------------+-----------+



In [0]:
!ls -l /dbfs/user/hive/warehouse/vuelos.db

total 0


## 10. Si borramos la tabla flightsjfk_externa, que se creó como externa directamente con saveAsTable, el borrado de la tabla no borrará los datos de ADLS

In [0]:
spark.sql("drop table vuelos.flightsjfk_externa")
spark.sql("show tables").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|        |weatherdistancetable|       true|
+--------+--------------------+-----------+



In [0]:
display(dbutils.fs.ls("abfss://datos@cursosparkucm.dfs.core.windows.net/flights"))

path,name,size,modificationTime
abfss://datos@cursosparkucm.dfs.core.windows.net/flights/_delta_log/,_delta_log/,0,1707125091000
abfss://datos@cursosparkucm.dfs.core.windows.net/flights/part-00000-e953baa0-62eb-44f8-ae16-c3631e493d94-c000.snappy.parquet,part-00000-e953baa0-62eb-44f8-ae16-c3631e493d94-c000.snappy.parquet,10761,1707125103000
abfss://datos@cursosparkucm.dfs.core.windows.net/flights/part-00001-531a06c9-2116-4c29-ba71-190ff0a7481b-c000.snappy.parquet,part-00001-531a06c9-2116-4c29-ba71-190ff0a7481b-c000.snappy.parquet,9984,1707125103000
abfss://datos@cursosparkucm.dfs.core.windows.net/flights/part-00002-f2a03258-75e5-429d-a913-80502b814f8e-c000.snappy.parquet,part-00002-f2a03258-75e5-429d-a913-80502b814f8e-c000.snappy.parquet,10402,1707125103000
abfss://datos@cursosparkucm.dfs.core.windows.net/flights/part-00003-939c3764-77ed-4c43-b019-4308c40cacc3-c000.snappy.parquet,part-00003-939c3764-77ed-4c43-b019-4308c40cacc3-c000.snappy.parquet,9335,1707125103000


In [0]:
dbutils.fs.mv("abfss://datos@cursosparkucm.dfs.core.windows.net/youtube_USvideos.csv", "abfss://datos@cursosparkucm.dfs.core.windows.net/datasets")

True

In [0]:
spark.sql("show tables").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|  vuelos|          flightssfo|      false|
|        |weatherdistancetable|       true|
+--------+--------------------+-----------+

